In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ks_2samp

# 1. Load Data Member M67
df = pd.read_csv("M67_Members_Final.csv")

# Pastikan semua angka
cols = ['ra', 'dec', 'ruwe']
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=cols)

# 2. Hitung Pusat Klaster (Berdasarkan Member)
center_ra = df['ra'].mean()
center_dec = df['dec'].mean()

# 3. Hitung Jarak Radial (r) dari Pusat
# Kita pakai koreksi cos(dec) agar jaraknya akurat dalam derajat
# Rumus: sqrt( (dRA * cos(Dec))^2 + dDec^2 )
d_ra = (df['ra'] - center_ra) * np.cos(np.radians(center_dec))
d_dec = df['dec'] - center_dec
df['radius_deg'] = np.sqrt(d_ra**2 + d_dec**2)

# Konversi ke Arcminutes (biar angkanya enak dilihat, 1 deg = 60 arcmin)
df['radius_arcmin'] = df['radius_deg'] * 60

# 4. Bagi Populasi: Single vs Binary (Berdasarkan RUWE)
# Ambang batas kita buat tegas agar kontras
# Single: RUWE < 1.1 (Sangat stabil)
# Binary: RUWE > 1.2 (Pasti goyang) -> Gap 1.1-1.2 kita buang biar bersih
single_stars = df[df['ruwe'] < 1.1]
binary_stars = df[df['ruwe'] > 1.2]

print(f"Jumlah Sample:")
print(f"   Single (RUWE < 1.1): {len(single_stars)}")
print(f"   Binary (RUWE > 1.2): {len(binary_stars)}")

# 5. Uji Statistik Kolmogorov-Smirnov (KS-Test)
# Hipotesis Nol: Kedua distribusi SAMA.
# Jika p-value < 0.05, Hipotesis Nol DITOLAK (Berarti BEDA SIGNIFIKAN).
stat, p_value = ks_2samp(single_stars['radius_arcmin'], binary_stars['radius_arcmin'])

print("\n--- HASIL STATISTIK (KS-TEST) ---")
print(f"KS Statistic: {stat:.4f}")
print(f"P-value     : {p_value:.2e}") # Format scientific (e.g., 1.23e-05)

if p_value < 0.05:
    print("KESIMPULAN: SIGNIFIKAN! Distribusi spasial mereka BERBEDA.")
else:
    print("KESIMPULAN: TIDAK SIGNIFIKAN. Mereka menyebar sama saja.")

# 6. Plotting CDF
plt.figure(figsize=(10, 7))

# Plot CDF Single (Biru)
plt.hist(single_stars['radius_arcmin'], bins=1000, density=True, cumulative=True, 
         histtype='step', color='blue', linewidth=2, label='Single (RUWE < 1.1)')

# Plot CDF Binary (Merah)
plt.hist(binary_stars['radius_arcmin'], bins=1000, density=True, cumulative=True, 
         histtype='step', color='red', linewidth=2, label='Binary (RUWE > 1.2)')

# Estetika
plt.xlabel('Distance from Center (arcmin)', fontsize=12)
plt.ylabel('Cumulative Fraction', fontsize=12)
plt.title(f'Mass Segregation in M67 (RUWE Proxy)\nKS-Test p-value: {p_value:.2e}', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right', fontsize=12)
plt.xlim(0, 60) # Fokus ke 1 derajat pertama (60 arcmin)


plt.show()